In [1]:
import os, sys, gc, subprocess, numpy as np, pandas as pd
from pathlib import Path

def _maybe_pip(path):
    if os.path.exists(path):
        try:
            print(f"[Install] {os.path.basename(path)}")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", path])
        except Exception as e:
            print(f"[Install skipped] {os.path.basename(path)} -> {e}")

RDKit_WHL_DIR = "/kaggle/input/rdkit-2025-3-3-cp311"
MORDRED_DIR   = "/kaggle/input/mordred-1-2-0-py3-none-any"

NX_WHL       = f"{MORDRED_DIR}/networkx-2.8.8-py3-none-any.whl"
MORDRED_WHL  = f"{MORDRED_DIR}/mordred-1.2.0-py3-none-any.whl"
# RDKit wheel
RDKit_WHL    = f"{RDKit_WHL_DIR}/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl"

_maybe_pip(NX_WHL)
_maybe_pip(RDKit_WHL)
_maybe_pip(MORDRED_WHL)

from rdkit import Chem
from mordred import Calculator, descriptors

def _find_data_dir():
    cands = [d for d in Path("/kaggle/input").glob("*")]
    key = lambda s: any(k in s.lower() for k in ["polymer", "neurips", "open", "opp"])
    for d in cands:
        if d.is_dir() and key(d.name.lower()):
            if (d/"train.csv").exists() and (d/"test.csv").exists():
                return str(d)
    return "/kaggle/input/neurips-open-polymer-prediction-2025"

DATA_DIR = _find_data_dir()
MODRED_DIR = "/kaggle/input/modred-dataset"
print("[DATA_DIR]", DATA_DIR)
print("[MODRED_DIR]", MODRED_DIR)

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

def _find_smiles(df):
    for c in df.columns:
        if str(c).lower().strip() in ("smiles","smile","smiles_str","polymer_smiles"):
            return c
    raise KeyError("SMILES column not found")

SMI_COL = _find_smiles(train)
TARGETS = ["Tg","FFV","Tc","Density","Rg"]

CACHE_TEST_DESC = "/kaggle/working/mordred_test.csv"

if os.path.exists(CACHE_TEST_DESC):
    desc_test = pd.read_csv(CACHE_TEST_DESC)
    desc_test = desc_test.loc[:, ~desc_test.columns.str.contains("^Unnamed")]
    print("[Cache] Loaded test descriptors:", desc_test.shape)
else:
    print("[Mordred] Computing test descriptors (2D; ignore_3D=True)...")
    mols_test = [Chem.MolFromSmiles(s) for s in test[SMI_COL].astype(str).tolist()]
    calc = Calculator(descriptors, ignore_3D=True)
    desc_test = calc.pandas(mols_test)  # returns DataFrame
    # Coerce numeric where possible, keep non-numeric as-is; we'll clean later
    desc_test = desc_test.apply(pd.to_numeric, errors="ignore")
    desc_test.to_csv(CACHE_TEST_DESC, index=False)
    print("[Cache] Saved test descriptors:", desc_test.shape)

def clean_desc_df(df, target_name=None):
    df = df.copy()
    for c in df.columns:
        if c != target_name:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    # Replace inf with NaN
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    # Drop columns with all NaN or single unique value (excluding target)
    nunique = df.nunique(dropna=True)
    drop_cols = [c for c in df.columns if c != target_name and (nunique.get(c, 2) <= 1)]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

USE_CATBOOST = True
CATBOOST_OK = False
if USE_CATBOOST:
    try:
        from catboost import CatBoostRegressor
        CATBOOST_OK = True
    except Exception as e:
        print("[CatBoost unavailable] Falling back to LightGBM:", e)
        USE_CATBOOST = False

if not USE_CATBOOST:
    from lightgbm import LGBMRegressor

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
N_SPLITS = 5

def fit_predict_target(target, desc_train_path, desc_test_df):
    print(f"\n=== [{target}] using {os.path.basename(desc_train_path)} ===")
    train_d = pd.read_csv(desc_train_path, low_memory=False)
    assert target in train_d.columns, f"{target} not found in {desc_train_path}"

    train_d = clean_desc_df(train_d, target_name=target)
    test_d  = clean_desc_df(desc_test_df)

    feat_cols = sorted(list((set(train_d.columns) - {target}) & set(test_d.columns)))
    X = train_d[feat_cols]
    y = train_d[target].astype(float)
    X_test = test_d[feat_cols]

    if not USE_CATBOOST:
        imp = SimpleImputer(strategy="median")
        X = pd.DataFrame(imp.fit_transform(X), columns=feat_cols)
        X_test = pd.DataFrame(imp.transform(X_test), columns=feat_cols)

    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(X), dtype=float)
    pred_test = np.zeros(len(X_test), dtype=float)

    for fold, (tr, va) in enumerate(kf.split(X), 1):
        Xtr, Xva = X.iloc[tr], X.iloc[va]
        ytr, yva = y.iloc[tr], y.iloc[va]

        if USE_CATBOOST:
            model = CatBoostRegressor(
                loss_function="MAE",
                depth=8,
                learning_rate=0.05,
                iterations=2000,   # early stopping will cut this down
                l2_leaf_reg=4.0,
                random_seed=RANDOM_STATE,
                verbose=False,
                od_type="Iter",
                od_wait=200,
            )
            model.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True, verbose=False)
        else:
            model = LGBMRegressor(
                objective="mae",
                n_estimators=3000,
                learning_rate=0.03,
                num_leaves=63,
                subsample=0.8,
                colsample_bytree=0.7,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
            )
            model.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="l1", verbose=False)

        oof[va] = model.predict(Xva)
        pred_test += model.predict(X_test) / N_SPLITS

        del model, Xtr, Xva, ytr, yva
        gc.collect()

    mae = float(mean_absolute_error(y, oof))
    print(f"[{target}] OOF MAE: {mae:.6f}  (features: {len(feat_cols)})")
    return pred_test

DESC_PATHS = {
    "Tg":      f"{MODRED_DIR}/desc_tg.csv",
    "FFV":     f"{MODRED_DIR}/desc_ffv.csv",
    "Tc":      f"{MODRED_DIR}/desc_tc.csv",
    "Density": f"{MODRED_DIR}/desc_de.csv",
    "Rg":      f"{MODRED_DIR}/desc_rg.csv",
}
DESC_PATHS = {t:p for t,p in DESC_PATHS.items() if os.path.exists(p)}

PRED = pd.DataFrame({"id": test["id"].values})
for t in TARGETS:
    if t in DESC_PATHS:
        PRED[t] = fit_predict_target(t, DESC_PATHS[t], desc_test)
    else:
        print(f"[{t}] descriptor CSV not found; filling zeros.")
        PRED[t] = 0.0

sub = pd.DataFrame({
    "id":      PRED["id"].values,
    "Tg":      PRED["Tg"].values,
    "FFV":     PRED["FFV"].values,
    "Tc":      PRED["Tc"].values,
    "Density": PRED["Density"].values,
    "Rg":      PRED["Rg"].values,
})
out_path = "/kaggle/working/submission.csv"
sub.to_csv(out_path, index=False)
print("[Saved]", out_path)

[Install] networkx-2.8.8-py3-none-any.whl


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
nx-cugraph-cu12 25.2.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-runtime-cu12==12.4.127; platform_

[Install] rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
[Install] mordred-1.2.0-py3-none-any.whl
[DATA_DIR] /kaggle/input/neurips-open-polymer-prediction-2025
[MODRED_DIR] /kaggle/input/modred-dataset
[Mordred] Computing test descriptors (2D; ignore_3D=True)...


100%|██████████| 3/3 [00:00<00:00,  5.28it/s]
/tmp/ipykernel_13/2508120266.py:65: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  desc_test = desc_test.apply(pd.to_numeric, errors="ignore")


[Cache] Saved test descriptors: (3, 1613)

=== [Tg] using desc_tg.csv ===
[Tg] OOF MAE: 25.489570  (features: 506)

=== [FFV] using desc_ffv.csv ===
[FFV] OOF MAE: 0.006271  (features: 506)

=== [Tc] using desc_tc.csv ===
[Tc] OOF MAE: 0.030611  (features: 506)

=== [Density] using desc_de.csv ===
[Density] OOF MAE: 0.073854  (features: 506)

=== [Rg] using desc_rg.csv ===
[Rg] OOF MAE: 1.640538  (features: 506)
[Saved] /kaggle/working/submission.csv
